# Lesson 2 — Modeling & Linearization

*ESP2110 Inverted Pendulum Lab*

**Run in Google Colab:** open the notebook, run the **Setup** cell once, then run
cells top-to-bottom. No local files are required.

## Learning objectives
By the end of this notebook you can:
1. Write the **nonlinear equations of motion** of the cart-pole and code them.
2. **Linearize** about the upright equilibrium (`sin theta ~ theta`, `cos theta ~ 1`) to get matrices `A`, `B`.
3. **Validate** the linear model against the nonlinear truth from the same initial condition.
4. Demonstrate **where linearization breaks** as the angle grows.

### Parameters (same plant used throughout the lab)
| Symbol | Meaning | Value |
| --- | --- | --- |
| `m_c` | Cart mass | 0.5 kg |
| `m_p` | Pole mass | 0.2 kg |
| `L` | Pole length | 0.3 m |
| `g` | Gravity | 9.81 m/s^2 |
| `dt` | Sample time | 0.01 s |

In [ ]:
# --- Setup (safe to re-run) ---
try:
    import numpy, scipy, matplotlib  # noqa: F401
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy', 'scipy', 'matplotlib'], check=True)
print('Environment ready.')

---
## The nonlinear equations of motion

With state `x = [p, v, theta, omega]`, input force `f`, and `theta` measured from upright:

$$\dot p = v,\qquad \dot\theta = \omega,$$
$$\dot v = \frac{f + m_p\sin\theta\,(L\omega^2 - g\cos\theta)}{m_c + m_p\sin^2\theta},$$
$$\dot\omega = \frac{-f\cos\theta - m_pL\omega^2\sin\theta\cos\theta + (m_c+m_p)g\sin\theta}{L\,(m_c + m_p\sin^2\theta)}.$$

These are **nonlinear** (products, `sin`, `cos`). That makes them realistic but hard to design
with — hence linearization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

m_c, m_p, L, g, dt = 0.5, 0.2, 0.3, 9.81, 0.01

def f_nonlin(s, f):
    """Full nonlinear cart-pole derivative. State s = [p, v, theta, omega]; theta=0 is upright."""
    p, v, th, om = s
    sin, cos = np.sin(th), np.cos(th)
    den = m_c + m_p * sin**2
    vdot  = (f + m_p * sin * (L * om**2 - g * cos)) / den
    omdot = (-f * cos - m_p * L * om**2 * sin * cos + (m_c + m_p) * g * sin) / (L * den)
    return np.array([v, vdot, om, omdot])

def simulate(force_fn, x0, T=2.0):
    """Euler-integrate the nonlinear plant. force_fn(k, x) -> applied force at step k."""
    n = int(T / dt); x = np.array(x0, dtype=float)
    X = np.zeros((n, 4)); F = np.zeros(n)
    for k in range(n):
        f = force_fn(k, x); F[k] = f; X[k] = x
        x = x + dt * f_nonlin(x, f)
    return np.arange(n) * dt, X, F

## Part 1 - Linearize about upright

Near `theta = 0` use `sin theta ~ theta`, `cos theta ~ 1`, and drop second-order terms
(`omega^2 theta`, `theta^2`, ...). The equations collapse to the linear system
`x_dot = A x + B f` with

$$A=\begin{bmatrix}0&1&0&0\\0&0&-\frac{m_pg}{m_c}&0\\0&0&0&1\\0&0&\frac{(m_c+m_p)g}{Lm_c}&0\end{bmatrix},
\qquad B=\begin{bmatrix}0\\\tfrac{1}{m_c}\\0\\-\tfrac{1}{Lm_c}\end{bmatrix}.$$

Fill in the two nonzero entries of `A` that come from gravity, and the two of `B`.

In [ ]:
# TODO: build the linearized A (4x4) and B (length-4) from the formulas above.
#       The gravity terms are A[1,2] = -m_p*g/m_c and A[3,2] = (m_c+m_p)*g/(L*m_c).
#       B = [0, 1/m_c, 0, -1/(L*m_c)]. Print both.
A = None  # <-- replace
B = None  # <-- replace


**Expected output.** `A[1,2] = -3.924`, `A[3,2] = 45.78`, `B = [0, 2, 0, -6.667]`. Only the
angle column of `A` is nonzero (gravity couples angle into the accelerations); position and
velocity do not appear because, to first order, they do not affect the dynamics near upright.

## Part 2 - Validate: linear vs nonlinear (small angle)

Run **both** models, unforced (`f = 0`), from the same small initial tilt and overlay the pole
angle. For small angles they should track closely — until the instability blows both up.

In [ ]:
# TODO: write sim_linear (uses A) and sim_nonlinear (uses f_nonlin), both unforced (f=0).
#       From theta0 = 5 deg, simulate 0.6 s and overlay the two pole-angle curves.


**Expected output.** At 5 deg the linear and nonlinear curves are **nearly indistinguishable** —
the linear model is an excellent local approximation. Both rise together (the upright is unstable),
but they rise *the same way*.

## Part 3 - Where linearization breaks

Repeat the comparison for a range of starting angles and measure the linear-vs-nonlinear
disagreement at a fixed early time (before the instability dominates).

In [ ]:
# TODO: for theta0 in {5,15,30,45} deg, simulate both models 0.4 s and print the
#       |linear - nonlinear| pole-angle error at t = 0.2 s. Watch it grow with angle.


**Expected output.** The error explodes with angle: about **0.04 deg at 5 deg**, **0.9 deg at
15 deg**, **5.6 deg at 30 deg**, and **14 deg at 45 deg** (by 0.4 s the 45 deg case is off by
>130 deg). The linear model is trustworthy only in a small neighbourhood of upright — which is
fine, because that is exactly the region a working controller keeps the pole in.

## Part 4 - Linear vs nonlinear, side by side (animation)

Two poles released from 30 deg: one evolved by the linear model, one by the nonlinear truth.
They start together and visibly drift apart.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

x0 = [0, 0, np.radians(30), 0.0]
t, XL = sim_linear(x0, 0.6); _, XN = sim_nonlinear(x0, 0.6)
frames = np.arange(0, len(XL), 3)
fig, ax = plt.subplots(figsize=(6, 3))
ax.set_xlim(-0.6, 0.6); ax.set_ylim(-0.1, 0.4); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('Linear (blue) vs nonlinear (red), theta0 = 30 deg'); ax.set_xlabel('x (m)')
linL, = ax.plot([], [], lw=3, color='steelblue', label='linear')
linN, = ax.plot([], [], lw=3, color='crimson', label='nonlinear'); ax.legend(loc='upper right')

def _u(j):
    i = frames[j]
    for X, ln, p0 in [(XL, linL, -0.0), (XN, linN, 0.0)]:
        th = X[i, 2]
        ln.set_data([p0, p0 + L * np.sin(th)], [0.0, L * np.cos(th)])
    return linL, linN

anim = animation.FuncAnimation(fig, _u, frames=len(frames), interval=70, blit=False)
plt.close(fig); HTML(anim.to_jshtml())

---
## Checkpoints
- `A[1,2] = -3.924`, `A[3,2] = 45.78`, `B = [0, 2, 0, -6.667]`.
- At 5 deg the linear and nonlinear angle curves overlap almost perfectly.
- The linear-vs-nonlinear error grows steeply with starting angle (~0.04 deg at 5 deg to ~14 deg at 45 deg by 0.2 s).
- You can explain *why* only the angle column of `A` is nonzero.

## Common pitfalls
- **Forgetting the small-angle assumption.** The linear model is only valid near `theta = 0`; don't trust it at large tilt.
- **Sign errors in `A`/`B`.** `A[1,2]` is negative, `A[3,2]` positive, `B[3]` negative — mixing them up flips the physics.
- **Comparing over too long a horizon.** The upright is unstable, so *both* models blow up eventually; judge the approximation early.
- **Radian/degree mixups** when printing or plotting angles.